In [5]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

def estimate_blur(image, threshold=200):
    laplacian = cv2.Laplacian(image, cv2.CV_64F)
    variance = laplacian.var()
    return variance, variance < threshold  

def estimate_noise(image):
    blurred = cv2.GaussianBlur(image, (5, 5), 1.5)
    diff = cv2.absdiff(image, blurred)
    noise_level = np.mean(diff)
    return noise_level, noise_level > 10  

def estimate_contrast(image):
    std_dev = np.std(image)
    hist = cv2.calcHist([image], [0], None, [256], [0, 256])
    non_zero = np.where(hist > 0)[0]
    if len(non_zero) > 0:
        spread = non_zero[-1] - non_zero[0]
    else:
        spread = 0
    
    low_contrast = (std_dev < 40) or (spread < 150)
    return std_dev, spread, low_contrast

def dynamic_preprocess(image):
    
    processed = image.copy()
    steps_applied = []
    
    blur_value, is_blurry = estimate_blur(image)
    print(f"Blur metric (Laplacian variance): {blur_value:.2f}")

    
    if is_blurry:
        kernel = np.array([[-1,-1,-1],
                          [-1, 9,-1],
                          [-1,-1,-1]])
        processed = cv2.filter2D(processed, -1, kernel)
        steps_applied.append("Sharpening")
        print("  → Image blurry: Applied sharpening")
    else:
        print("  → Image is sharp enough")
    
    noise_level, is_noisy = estimate_noise(processed)
    print(f"Noise level: {noise_level:.2f}")


    
    if is_noisy:
  
        if noise_level > 20:
            kernel_size = (7, 7)
        elif noise_level > 15:
            kernel_size = (5, 5)
        else:
            kernel_size = (3, 3)
        
        
        processed = cv2.GaussianBlur(processed, kernel_size, 1.0)
        steps_applied.append(f"Gaussian blur {kernel_size}")
        
        print(f"  → Noisy: Applied {steps_applied[-1]}")
    else:
        print("  → Noise level acceptable")

    
    
    std_dev, spread, low_contrast = estimate_contrast(processed)
    print(f"Contrast (std dev): {std_dev:.2f}, Histogram spread: {spread}")
    
    if low_contrast:
        if is_noisy or is_blurry:
            clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
            processed = clahe.apply(processed)
            steps_applied.append("CLAHE equalization")
        else:
            processed = cv2.equalizeHist(processed)
            steps_applied.append("Global histogram equalization")
        
        print(f"  → Low contrast: Applied {steps_applied[-1]}")
    else:
        print("  → Contrast is good")
    
    return processed, steps_applied


    
count = 0
while True:
    path = "C:/Users/USER/Desktop/image processing project/image processing video/raveling/extracted/"+ str(count) +".jpg"
    img = cv2.imread(path, 0)
    
    
    if img is None:
        print("No frame")
        break
    else:
        processed_img, steps = dynamic_preprocess(img)
        
        cv2.imshow("processed",processed_img)
        cv2.imshow("original",img)
        if cv2.waitKey(100)==ord('q'):
            break
        
    count = count + 5

cv2.destroyAllWindows()





Blur metric (Laplacian variance): 9137.84
  → Image is sharp enough
Noise level: 20.97
  → Noisy: Applied Gaussian blur (7, 7)
Contrast (std dev): 24.65, Histogram spread: 214
  → Low contrast: Applied CLAHE equalization
Blur metric (Laplacian variance): 3003.09
  → Image is sharp enough
Noise level: 15.03
  → Noisy: Applied Gaussian blur (5, 5)
Contrast (std dev): 24.90, Histogram spread: 218
  → Low contrast: Applied CLAHE equalization
Blur metric (Laplacian variance): 4108.88
  → Image is sharp enough
Noise level: 16.64
  → Noisy: Applied Gaussian blur (5, 5)
Contrast (std dev): 25.83, Histogram spread: 230
  → Low contrast: Applied CLAHE equalization
Blur metric (Laplacian variance): 3352.42
  → Image is sharp enough
Noise level: 15.96
  → Noisy: Applied Gaussian blur (5, 5)
Contrast (std dev): 26.21, Histogram spread: 228
  → Low contrast: Applied CLAHE equalization
Blur metric (Laplacian variance): 3647.32
  → Image is sharp enough
Noise level: 16.49
  → Noisy: Applied Gaussian b

In [ ]:

# Display results
plt.figure(figsize=(15, 8))

plt.subplot(2, 3, 1)
plt.imshow(img, cmap='gray')
plt.title("Original Image")
plt.axis('off')

plt.subplot(2, 3, 2)
plt.imshow(processed_img, cmap='gray')
plt.title(f"Dynamic Processed\nSteps: {', '.join(steps) if steps else 'None'}")
plt.axis('off')

plt.subplot(2, 3, 3)
hist_original = cv2.calcHist([img], [0], None, [256], [0, 256])
plt.plot(hist_original, color='red')
plt.title("Original Histogram")
plt.xlim([0, 256])

plt.subplot(2, 3, 4)
hist_processed = cv2.calcHist([processed_img], [0], None, [256], [0, 256])
plt.plot(hist_processed, color='blue')
plt.title("Processed Histogram")
plt.xlim([0, 256])





plt.tight_layout()
plt.show()



